# Démo 2 · Interroger les données de la NHL avec PandasAI + Qwen en local

Un **LLM** génère du texte ou du code à partir d’un prompt. **PandasAI** relie ce modèle à une table pandas : il construit un prompt d’analyse, obtient du code et l’exécute pour répondre à notre question.

Nous utilisons **PandasAI 2.3.2**, son `SmartDataframe` et un petit adaptateur pour **Qwen2.5-1.5B-Instruct**, exécuté avec Transformers.

**Question en langage naturel → Qwen génère du Python → PandasAI l’exécute → examiner et vérifier**

PandasAI peut renvoyer un nombre, une table ou un graphique. Il peut aussi générer un calcul plausible, mais incorrect. Nous comparerons trois tâches avec des calculs pandas ordinaires à partir de **MTL–CAR, le 21 mai 2026 (`2025030311`)**.

Ce notebook est autonome. La [partie 4](04_llm_and_rag.ipynb) récupère de la *documentation d’API*; ici, nous interrogeons *une table déjà chargée*. PandasAI n’utilise pas automatiquement l’index de documentation de la partie 4.


## Configuration : choisir l’environnement local ou Colab

**Développement local (voir le [matériel conseillé](../../README.md#matériel-conseillé-pour-les-notebooks-04-et-05)) :** lancez `uv sync --group llm` à la racine du dépôt et sélectionnez le noyau de l’environnement commun `.venv` sous Python 3.11. **Sautez la cellule d’installation facultative** et passez à la vérification de l’environnement ci-dessous. Il n’est pas nécessaire d’installer les paquets individuellement.

**Google Colab | sans clonage du dépôt ni installation manuelle de uv :**

1. Ouvrez ce notebook dans Colab. Dans **Runtime → Change runtime type**, choisissez **T4 GPU** et le **runtime 2025.07 (Python 3.11)** s’il est disponible. Le runtime par défaut peut utiliser une version plus récente de Python; la cellule d’installation ne peut pas changer l’interpréteur en cours d’exécution. Si aucun runtime Python 3.11 n’est disponible, utilisez l’environnement local. [Versions des runtimes Colab](https://research.google.com/colaboratory/runtime-version-faq.html)
2. Dans la **cellule facultative ci-dessous**, définissez `INSTALL_COLAB_PACKAGES = True` et exécutez-la une fois. La cellule installe `uv`, puis l’utilise pour installer les bibliothèques dans le Python actuel du notebook. `sys.executable` est le chemin de ce Python; `subprocess.check_call` lance une commande et s’arrête si elle échoue. Colab fournit déjà PyTorch; la cellule installe les dépendances supplémentaires.
3. Choisissez **Runtime → Restart session** après l’installation, même si aucun avertissement de redémarrage ne s’affiche. C’est nécessaire, car NumPy/pandas peuvent déjà être chargés dans des versions différentes.
4. Remettez le paramètre à `False`, puis exécutez la **vérification de l’environnement** et passez à **Charger Qwen**. Répétez l’installation lorsque Colab vous attribue un nouveau runtime; un simple redémarrage de session conserve les paquets installés.

Le premier téléchargement du modèle nécessite Internet; l’inférence s’exécute ensuite dans votre propre runtime, sans clé d’API. Le mode CPU fonctionne, mais il est plus lent. Pour exécuter du code généré, utilisez un nouveau runtime Colab contenant uniquement des fichiers publics de la NHL : ne montez pas Drive et n’ajoutez pas d’identifiants de connexion. Pour exécuter du code généré en local, utilisez un environnement jetable sans accès aux fichiers personnels; un environnement virtuel Python ne constitue pas à lui seul un bac à sable.


In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync --group llm in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==1.5.3",
        "transformers==4.57.6",
        "accelerate>=1,<2",
        "bitsandbytes>=0.45,<1",
        "requests",
        "pandasai==2.3.2",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")


### Après l’installation : vérifier l’environnement

Dans Colab, **redémarrez d’abord la session** et sautez la cellule d’installation lors de l’exécution suivante. En local, effectuez cette vérification après `uv sync --group llm`. Importer NumPy et pandas ici vérifie aussi que leurs binaires installés peuvent être chargés ensemble.


In [ ]:
import sys

if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Use a Python 3.11 kernel: the course .venv locally, or a compatible Colab runtime.")

import numpy as np
import pandas as pd
import torch
from importlib.metadata import version

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "| pandas:", pd.__version__)
print("Transformers:", version("transformers"))
print("PandasAI:", version("pandasai"))
print("CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unavailable — CPU/Apple GPU fallback")

### Charger Qwen

Nous utilisons le pipeline de Hugging Face et le gabarit de conversation de Qwen. Le chargement en **4 bits** stocke des poids compressés sur un GPU CUDA; les exécutions sur CPU et GPU Apple utilisent des poids ordinaires. `do_sample=False` utilise un décodage glouton; les résultats peuvent tout de même varier selon les appareils et les versions des paquets. Le code de configuration est fourni : concentrez-vous sur les prompts et leurs résultats.

**Qu’importons-nous ?**

- **`torch` (PyTorch)** effectue les calculs numériques du modèle sur CPU ou GPU.
- **`transformers`** est la bibliothèque de Hugging Face qui permet de charger et d’utiliser des modèles préentraînés.
- **`AutoTokenizer`** charge le tokenizer de Qwen : il convertit le texte en *tokens* numérotés (mots ou fragments de mots) que le modèle peut traiter, puis reconvertit les tokens générés en texte.
- **`AutoModelForCausalLM`** charge le modèle de génération de texte. « Causal » signifie qu’il prédit le prochain token à partir des tokens précédents.
- **`BitsAndBytesConfig`** décrit comment compresser les poids du modèle en valeurs sur 4 bits pour économiser la mémoire du GPU. Il n’entraîne pas le modèle.
- **`pipeline`** relie le tokenizer et le modèle pour fournir un outil pratique de génération de texte. Cet utilitaire de Hugging Face est une composante de notre processus global d’analyse des données.

**Lire la configuration :** `MODEL_ID` sélectionne le modèle; `from_pretrained(...)` télécharge ses fichiers sauvegardés à la première exécution et réutilise ensuite le cache de téléchargement. `device` choisit un GPU NVIDIA (`cuda`), un GPU Apple (`mps`) ou le CPU. Le `dtype` numérique et les options de quantification contrôlent le stockage du modèle et les calculs effectués.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = True
device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_options = {"dtype": torch.float32 if device == "cpu" else torch.float16}
if device == "cuda":
    model_options["device_map"] = "auto"
    if USE_4BIT:
        model_options["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_options)
if device != "cuda":
    model = model.to(device)
gen_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def ask_qwen(prompt, system="You are a helpful assistant.", max_new_tokens=512):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return gen_pipe(text, max_new_tokens=max_new_tokens, do_sample=False,
                    return_full_text=False, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"].strip()

print("Loaded", MODEL_ID, "on", device)

**Comment fonctionne `ask_qwen()` :**

1. Construire deux messages : `system` donne les consignes générales; `user` contient notre question ou notre tâche.
2. `apply_chat_template` ajoute les marqueurs de rôle et la mise en forme attendus par Qwen. Cela l’aide à distinguer les consignes du message de l’utilisateur.
3. `gen_pipe` découpe le texte en tokens, génère des tokens, puis les reconvertit en texte. `max_new_tokens` limite la longueur de la réponse; `do_sample=False` choisit le prochain token le plus probable à chaque étape.
4. `return_full_text=False` renvoie la nouvelle réponse sans répéter le prompt d’entrée. La fonction extrait `generated_text` du résultat du pipeline.

Nous pouvons réutiliser cette même fonction pour obtenir une explication ou une proposition de code : le prompt détermine la tâche.


## 2. Charger un match et préparer notre petite table

Nous reprenons les étapes essentielles de la partie 1. Le chemin du cache est relatif au répertoire de travail du notebook. Pour éviter un nouveau téléchargement, copiez le JSON de la partie 1 à cet endroit. Si l’API de la NHL est indisponible, utilisez le fichier mis en cache par la personne qui enseigne.

**Ce que fait la cellule de téléchargement :**

- **`requests`** envoie une requête GET à l’API de la NHL; **`Path`** gère le nom du fichier de cache local et les dossiers.
- Si le fichier existe, nous le réutilisons. Sinon, `raise_for_status()` vérifie les erreurs HTTP et les assertions vérifient l’identifiant du match et la liste des événements avant la sauvegarde.
- **`json.dumps`** transforme l’objet Python téléchargé en texte JSON pour le stocker; **`json.loads`** reconvertit ce texte en objet Python. À la fin, `game` est un dictionnaire contenant les données du match.


In [ ]:
import json
from pathlib import Path
import requests

GAME_ID = 2025030311
raw_path = Path("data/raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30,
    )
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(json.dumps(downloaded), encoding="utf-8")
game = json.loads(raw_path.read_text(encoding="utf-8"))
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

### Du JSON imbriqué à une table

- **`pandas` (`pd`)** fournit le DataFrame : une table avec des colonnes nommées. `pd.json_normalize(game["plays"])` aplatit les champs imbriqués des événements en colonnes comme `details.shotType`.
- `.isin(...)` construit un filtre pour les tirs et les buts; `.loc[...]` sélectionne ces lignes et les colonnes voulues. `.rename(...)` donne des noms plus courts aux colonnes et `.copy()` crée une table distincte à modifier.
- `.map(team_names)` remplace les identifiants numériques des équipes par des abréviations. `.eq("goal")` crée la colonne booléenne `is_goal`.
- Nous conservons quatre colonnes, réinitialisons l’index des lignes et vérifions les identifiants d’événements en double et les équipes manquantes. **`display`** affiche un aperçu dans le notebook; **`Image`** affichera plus tard le graphique généré.


In [ ]:
import pandas as pd
from IPython.display import display, Image

events = pd.json_normalize(game["plays"])
columns = {"eventId": "event_id", "typeDescKey": "event_type",
           "details.eventOwnerTeamId": "team_id", "details.shotType": "shot_type"}
shots = events.loc[events["typeDescKey"].isin(["shot-on-goal", "goal"]), list(columns)].rename(columns=columns).copy()
team_names = {game[side]["id"]: game[side]["abbrev"] for side in ["awayTeam", "homeTeam"]}
shots["team"] = shots["team_id"].map(team_names)
shots["is_goal"] = shots["event_type"].eq("goal")
shots = shots[["event_id", "team", "shot_type", "is_goal"]].reset_index(drop=True)
assert shots["event_id"].is_unique and shots["team"].notna().all()
display(shots.head())

**Définissez « tir » avant d’interroger le modèle.** Chaque ligne représente un tir au but **ou un but**. Les tirs manqués et bloqués sont exclus. Notre pourcentage de buts est `100 × goals / all rows`, calculé par équipe. Les types de tirs manquants restent manquants et ne retirent pas de lignes des totaux par équipe.

Calculez d’abord les réponses de référence. Pour les données utilisées en classe : MTL compte **22 tirs et 6 buts**; CAR compte **28 tirs et 2 buts**. Ce sont des points de vérification, pas des valeurs à coder en dur. Ce match ne fait pas partie des saisons demandées dans le jalon et ne permet pas de tirer des conclusions sur une saison entière.

**Lire le calcul de référence :** `groupby("team")` sépare les lignes selon l’équipe qui tire. Dans `agg`, `size` compte toutes les lignes et `sum` additionne les valeurs booléennes de `is_goal` (`True` vaut 1, `False` vaut 0). Nous calculons ensuite le pourcentage à partir de ces deux totaux. `expected` est notre table de référence pour vérifier le travail du modèle.


In [ ]:
expected = shots.groupby("team").agg(shots=("event_id", "size"), goals=("is_goal", "sum"))
expected["goal_pct"] = 100 * expected["goals"] / expected["shots"]
display(expected)

## 3. Relier Qwen à PandasAI

Il s’agit du même patron d’adaptateur que dans l’ébauche. Nous demandons explicitement au petit modèle d’utiliser `dfs[0]` (la vraie table), plutôt que de reconstruire les données à partir des lignes d’exemple de son prompt. PandasAI construit un prompt d’analyse; notre méthode `call()` le transmet au gabarit de conversation de Qwen. L’adaptateur ne contient aucune logique d’analyse de la NHL.

PandasAI 2 renvoie directement la réponse de `.chat()`; examinez le code dans `sdf.last_code_executed`. Ses vérifications de code **ne constituent pas un bac à sable de sécurité**. Utilisez l’environnement jetable décrit plus haut pour ces cellules.

**À quoi servent ces classes ?**

- **`SmartDataframe`** enveloppe notre table pandas ordinaire et ajoute `.chat()`. Nous conservons aussi `shots` pour l’analyse manuelle.
- **`LLM`** est l’interface de base de PandasAI pour un modèle de langage. `QwenPandasLLM(LLM)` implémente cette interface pour notre modèle Qwen local; il ne crée ni n’entraîne de nouveau modèle.
- **`call()`** reçoit les consignes de PandasAI, les convertit en texte et les envoie à `ask_qwen()`. Le texte renvoyé est le code d’analyse proposé. `type` est simplement une étiquette identifiant cet adaptateur.
- **`self.last_prompt` / `self.last_response`** conservent la dernière entrée et la sortie brute du modèle pour les examiner. `calls` compte le nombre de demandes de génération de texte à Qwen, y compris les éventuelles demandes internes de correction.

**Relier les éléments :** `pandas_llm` est une instance de notre adaptateur; `config["llm"]` indique à `SmartDataframe` de l’utiliser. `dfs[0]` est le nom que PandasAI donne au premier DataFrame fourni pendant l’exécution du code. `plot_dir` est le dossier où les graphiques générés sont sauvegardés, et `data_context` explique nos colonnes et nos définitions de hockey au modèle.

`enable_cache=False` désactive la réutilisation des réponses de PandasAI. Il ne désactive **pas** le cache JSON de la NHL créé auparavant. Ce sont deux caches distincts pour deux étapes différentes.


In [ ]:
from pandasai import SmartDataframe
from pandasai.llm.base import LLM

class QwenPandasLLM(LLM):
    calls = 0

    @property
    def type(self):
        return "qwen-local"

    def call(self, instruction, context=None):
        self.calls += 1
        self.last_prompt = instruction.to_string() + (
            "\nUse the existing df = dfs[0]. Complete the calculation in the QUERY. "
            "Replace all TODOs and placeholder text with working Python. "
            "Do not read another file or recreate the sample data."
        )
        self.last_response = ask_qwen(
            self.last_prompt,
            system="You are a data analyst. Return executable pandas code in a fenced python block. "
                   "Follow the requested result dictionary format.",
            max_new_tokens=768,
        )
        return self.last_response

plot_dir = Path("data/plots").resolve()
plot_dir.mkdir(parents=True, exist_ok=True)
pandas_llm = QwenPandasLLM()
sdf = SmartDataframe(shots, config={
    "llm": pandas_llm, "enable_cache": False, "verbose": False,
    "save_logs": False, "max_retries": 1, "use_error_correction_framework": False,
    "save_charts": True, "save_charts_path": str(plot_dir), "open_charts": False,
})
data_context = (
    "This table contains only game 2025030311, MTL vs CAR on 2026-05-21. "
    "Each row is one shot INCLUDING goals. is_goal is Boolean. "
    "Use df = dfs[0] directly; it is already filtered to this game. "
    "The only columns are event_id, team, shot_type, is_goal. "
    "Missing shot_type does not exclude a row. "
)

## 4. Trois questions, trois vérifications

Commencez par un dénombrement, puis une agrégation et enfin un graphique. Un message d’erreur ou un résultat inutilisable compte comme une tentative échouée. La mise en cache des réponses est désactivée. PandasAI peut tout de même effectuer des appels internes de correction; nous comptons donc aussi les appels au modèle. Si vous révisez un prompt, consignez la nouvelle tentative séparément.

**D’abord, demandez un nombre :**

- `data_context + ...` joint nos définitions des données à la question, afin que le modèle sache ce que représente une ligne.
- `sdf.chat(..., output_type="number")` demande à PandasAI de générer et d’exécuter du code d’analyse donnant un résultat numérique. Préciser le type de sortie ne garantit pas que le calcul est correct.
- **`Number`** permet de vérifier que la réponse est numérique. Nous la comparons ensuite à la valeur de référence dans `expected.loc["MTL", "goals"]` et stockons le résultat de la comparaison dans `count_ok`. Afficher le code aide à expliquer un écart.


In [ ]:
from numbers import Number

count_prompt = data_context + "How many rows have team == 'MTL' and is_goal == True? Return a number."
count_answer = sdf.chat(count_prompt, output_type="number")
count_code = sdf.last_code_executed
print("Answer:", count_answer)
print("Generated code:\n", count_code)
count_ok = isinstance(count_answer, Number) and count_answer == int(expected.loc["MTL", "goals"])
print("Matches pandas:", count_ok)

### Demander une table, puis comparer son contenu

- Le prompt précise les colonnes de sortie et le regroupement pour que le résultat à vérifier soit clair.
- PandasAI 2 peut envelopper sa réponse dans un `SmartDataframe`; `.dataframe` extrait la table pandas sous-jacente.
- Avant de comparer, nous utilisons les noms des équipes comme index, trions les deux tables et sélectionnons les mêmes colonnes. Cela évite de considérer un ordre différent des équipes comme une erreur.
- **`pd.testing.assert_frame_equal`** compare les valeurs à notre référence. Les options autorisent des dtypes numériques différents et de minuscules écarts d’arrondi. `try` / `except` consigne une comparaison échouée avec `table_ok=False`, afin de pouvoir examiner le problème et poursuivre.


In [ ]:
table_prompt = data_context + (
    "Return a DataFrame with exactly three columns: team, shots (count of rows), "
    "goals (sum of is_goal). Group by team so there is one row per team."
)
table_answer = sdf.chat(table_prompt, output_type="dataframe")
table_code = sdf.last_code_executed
display(table_answer)
print(table_code)
# PandasAI 2 wraps tabular answers in a SmartDataframe.
actual = table_answer.dataframe if isinstance(table_answer, SmartDataframe) else table_answer
try:
    pd.testing.assert_frame_equal(
        actual.set_index("team").sort_index()[["shots", "goals"]], expected[["shots", "goals"]].sort_index(),
        check_dtype=False, check_exact=False, rtol=1e-6, atol=1e-6,
    )
    table_ok = True
except (AssertionError, AttributeError, KeyError, TypeError) as error:
    table_ok = False
    print("Mismatch to investigate:", error)
print("Matches pandas:", table_ok)

**Examinez le calcul :** le code généré compte-t-il toutes les lignes comme des tirs, additionne-t-il le booléen `is_goal` et regroupe-t-il les résultats par équipe ? Supprime-t-il par erreur les lignes dont le type de tir est manquant ? Une explication bien formulée ne répond pas à ces questions.

**Ensuite, demandez un graphique :**

- `output_type="plot"` demande du code de visualisation. Python trace le graphique et sauvegarde une image; Qwen génère les instructions, pas les pixels de l’image.
- La réponse renvoyée peut être un chemin vers un PNG. **`Image`** et **`display`** de `IPython.display` affichent ce fichier dans le notebook; si aucune image utilisable n’est renvoyée, nous affichons plutôt la réponse.
- Examinez le code généré et comparez les hauteurs des barres à `expected["goals"]`. Un titre qui semble correct ou un joli graphique ne suffit pas à valider les valeurs.


In [ ]:
chart_prompt = data_context + (
    "Create a bar chart of total goals for each team: group by team and sum is_goal. "
    "Label the axes Team and Goals. Title: MTL vs CAR — one game. "
    "The output folder already exists; do not create directories."
)
chart_answer = sdf.chat(chart_prompt, output_type="plot")
chart_code = sdf.last_code_executed
if isinstance(chart_answer, str) and chart_answer.endswith(".png") and Path(chart_answer).is_file():
    display(Image(filename=chart_answer))
else:
    print(chart_answer)
print(chart_code)
display(expected[["goals"]])  # Compare these values with the bar heights.

## 5. Évaluer, puis essayer une question à laquelle les données ne permettent pas de répondre

Ne marquez le graphique comme correct que si ses valeurs, ses étiquettes et la présence des deux équipes correspondent à la référence. Trois tâches constituent une vérification en classe, pas un benchmark du modèle. Conservez les prompts et le code généré, y compris les tentatives échouées.

Si Qwen ne produit aucun graphique utilisable, la référence manuelle `expected["goals"].plot.bar(xlabel="Team", ylabel="Goals")` permet de poursuivre la discussion. Indiquez qu’elle est manuelle et marquez la **tentative de graphique du LLM** comme échouée. Lors de notre préparation locale, le dénombrement a réussi, mais des erreurs de format de table et de tracé ont persisté; le tutoriel ne suppose pas que les trois réponses réussissent.

**Lire la cellule d’évaluation :** `count_ok` et `table_ok` proviennent de comparaisons automatiques; `chart_ok` commence à `None`, car il nécessite votre vérification visuelle. Une fois les trois résultats renseignés, convertir les booléens en nombres et calculer leur moyenne donne la proportion de tâches réussies. Le compteur d’appels au modèle est distinct : une tâche peut nécessiter plusieurs générations.


In [ ]:
chart_ok = None  # Replace with True or False after inspecting the chart; an error is False.
evaluation = pd.DataFrame({
    "task": ["MTL goal count", "Team summary", "Goals-by-team chart"],
    "correct": [count_ok, table_ok, chart_ok],
})
display(evaluation)
print("Model calls, including internal repairs:", pandas_llm.calls)
if evaluation["correct"].notna().all():
    print(f"Success rate: {evaluation['correct'].astype(bool).mean():.0%}")
else:
    print("Review the chart before reporting a success rate.")

**Dépannage :** si le code est tronqué, augmentez `max_new_tokens` dans l’adaptateur. Si l’extraction ou l’exécution échoue, examinez le prompt et le code, puis simplifiez la question; ne supposez pas que le petit modèle peut résoudre toutes les tâches. La génération sur CPU peut prendre beaucoup plus de temps que sur GPU.

**Sources :** [Fiche du modèle Qwen](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) · [PandasAI 2.3.2](https://pypi.org/project/pandasai/2.3.2/). Ce notebook suit volontairement **l’API v2** de l’ébauche, et non les exemples v3 trouvés ailleurs.
